# SAP HANA Cloud Knowledge Graph Engine

>[SAP HANA Cloud Knowledge Graph](https://help.sap.com/docs/hana-cloud-database/sap-hana-cloud-sap-hana-database-knowledge-graph-guide/sap-hana-cloud-sap-hana-database-knowledge-graph-engine-guide) is a fully integrated knowledge graph solution within the `SAP HANA Cloud` database.
>
>This example demonstrates how to build a QA (Question-Answering) chain that queries [Resource Description Framework (RDF)](https://en.wikipedia.org/wiki/Resource_Description_Framework) data stored in an `SAP HANA Cloud` instance using the `SPARQL` query language, and returns a human-readable response.
>
>[SPARQL](https://en.wikipedia.org/wiki/SPARQL) is the standard query language for querying `RDF` graphs.


## Setup & Installation

**Prerequisite**:  
You must have an SAP HANA Cloud instance with the **triple store** feature enabled.  
For detailed instructions, refer to: [Enable Triple Store](https://help.sap.com/docs/hana-cloud-database/sap-hana-cloud-sap-hana-database-knowledge-graph-guide/enable-triple-store)

To use SAP HANA Knowledge Graph Engine and/or Vector Store Engine with LangChain, install the `langchain-hana` package:

In [ ]:
%pip install -qU langchain-hana

First, create a connection to your SAP HANA Cloud instance.

In [1]:
from dotenv import load_dotenv
from hdbcli import dbapi
import os

# Load environment variables if needed
load_dotenv()

# Establish connection to SAP HANA Cloud
connection = dbapi.connect(
    address=os.environ.get("HANA_DB_ADDRESS"),
    port=os.environ.get("HANA_DB_PORT"),
    user=os.environ.get("HANA_DB_USER"),
    password=os.environ.get("HANA_DB_PASSWORD"),
    autocommit=True,
    sslValidateCertificate=False,
)

## Example: Question Answering over a “Movies” Knowledge Graph

Below we’ll:

1. Instantiate the `HanaRdfGraph` pointing at our “movies” data graph and its ontology  
2. Wrap it in a `HanaSparqlQAChain` powered by an LLM  
3. Ask natural-language questions and print out the chain’s responses  

This demonstrates how the LLM generates SPARQL under the hood, executes it against SAP HANA, and returns a human-readable answer.


In [2]:
from langchain_hana import HanaRdfGraph, HanaSparqlQAChain
from langchain_openai import ChatOpenAI  # or your chosen LLM

In [3]:
# Set up the Knowledge Graph
graph_uri = "http://kg.demo.sap.com/movies"
ontology_uri = "http://kg.demo.sap.com/movies_ontology"
graph = HanaRdfGraph(
    connection=connection,
    graph_uri=graph_uri,
    ontology_uri=ontology_uri
)

In [4]:
# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o")

In [5]:
# Create a SPARQL QA Chain
chain = HanaSparqlQAChain.from_llm(
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True,
    graph=graph)

In [6]:
output = chain.invoke("In which movies did Harrison Ford and Liam Neeson play in together")
print(output['result'])



> Entering new HanaSparqlQAChain chain...
Generated SPARQL:
```sparql
PREFIX kg: <http://kg.demo.sap.com/>

SELECT ?movieTitle
WHERE {
    ?actor1 kg:name "Harrison Ford" .
    ?actor2 kg:name "Liam Neeson" .
    ?movie kg:title ?movieTitle .
    ?actor1 kg:acted_in ?movie .
    ?actor2 kg:acted_in ?movie .
}
```
Final SPARQL:

PREFIX kg: <http://kg.demo.sap.com/>

SELECT ?movieTitle

FROM <http://kg.demo.sap.com/movies>
WHERE {
    ?actor1 kg:name "Harrison Ford" .
    ?actor2 kg:name "Liam Neeson" .
    ?movie kg:title ?movieTitle .
    ?actor1 kg:acted_in ?movie .
    ?actor2 kg:acted_in ?movie .
}

Full Context:
movieTitle
K-19: The Widowmaker
Anchorman 2: The Legend Continues


> Finished chain.
Harrison Ford and Liam Neeson both appeared in the movie "K-19: The Widowmaker."


In [8]:
output = chain.invoke("What are the three movies with the highest budgets, their titles, and the names of the directors who directed them?")
print(output['result'])



> Entering new HanaSparqlQAChain chain...
Generated SPARQL:
```sparql
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX : <http://kg.demo.sap.com/>

SELECT ?title ?budget ?directorName
WHERE {
    ?movie a :Movie ;
           :title ?title ;
           :budget ?budget .
    ?director a :Director ;
              :directed ?movie ;
              :name ?directorName .
}
ORDER BY DESC(?budget)
LIMIT 3
```
Final SPARQL:

PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX : <http://kg.demo.sap.com/>

SELECT ?title ?budget ?directorName

FROM <http://kg.demo.sap.com/movies>
WHERE {
    ?movie a :Movie ;
           :title ?title ;
           :budget ?budget .
    ?director a :Director ;
              :directed ?movie ;
              :name ?directorName .
}
ORDER BY DESC(?budget)
LIMIT 3

Full Context:
title,budget,directorName
Pirates of the Caribbean: On Stranger Tides,380000000,Rob Marshall
Pirates of the Caribbean: At World's End,300000000,Gore Verbinski
Avengers: Age of Ultron,2

### What’s happening under the hood?

1. **SPARQL Generation**  
   The chain invokes the LLM with your Turtle-formatted ontology (`graph.get_schema`) and the user’s question using the `SPARQL_GENERATION_SELECT_PROMPT`. The LLM then emits a valid `SELECT` query tailored to your schema.

2. **Pre-processing & Execution**  
   - **Extract & clean**: Pull the raw SPARQL text out of the LLM’s response.  
   - **Inject graph context**: Add `FROM <graph_uri>` if it’s missing and ensure common prefixes (`rdf:`, `rdfs:`, `owl:`, `xsd:`) are declared.  
   - **Run on HANA**: Execute the finalized query via `HanaRdfGraph.query()` over your named graph.

3. **Answer Formulation**  
   The returned CSV (or Turtle) results feed into the LLM again—this time with the `SPARQL_QA_PROMPT`. The LLM produces a concise, human-readable answer strictly based on the retrieved data, without hallucination.


## Initialize the `HanaRdfGraph`

To power the QA chain, you first need a `HanaRdfGraph` instance that:

1. Loads your ontology schema (in Turtle)  
2. Executes SPARQL queries against your SAP HANA Cloud data graph  

The constructor requires:

- **`connection`**: an active `hdbcli.dbapi.connect(...)` instance  
- **`graph_uri`**: the named graph (or `"DEFAULT"`) where your RDF data lives  
- **One of**:  
  - **`ontology_query`**: a SPARQL CONSTRUCT to extract schema triples  
  - **`ontology_uri`**: a hosted ontology graph URI  
  - **`ontology_local_file`** + **`ontology_local_file_format`**: a local Turtle/RDF file  
  - **`auto_extract_ontology=True`** (not recommended for production—see note)

### `graph_uri` vs. Ontology
- **`graph_uri`**:  
  The named graph in your SAP HANA Cloud instance that contains your instance data (sometimes 100k+ triples).
  If `None` or `"DEFAULT"` is provided, the default graph is used.  
  ➔ More details: [Default Graph and Named Graphs](https://help.sap.com/docs/hana-cloud-database/sap-hana-cloud-sap-hana-database-knowledge-graph-guide/default-graph-and-named-graphs)
- **Ontology**: a lean schema (typically ~50-100 triples) describing classes, properties, domains, ranges, labels, comments, and subclass relationships. The ontology guides SPARQL generation and result interpretation.


1. **Custom SPARQL CONSTRUCT query** (`ontology_query`): Use a custom `CONSTRUCT` query to selectively extract schema triples.


In [9]:
custom_construct = """
CONSTRUCT {
  ?s ?p ?o .
}
FROM <http://kg.demo.sap.com/movies_ontology>
WHERE {
    ?s ?p ?o .
}
"""
graph = HanaRdfGraph(
 connection=connection,
 graph_uri="http://kg.demo.sap.com/movies",
 ontology_query=custom_construct
)

2. **Remote ontology URI** (`ontology_uri`): Load the schema directly from a hosted graph URI.

In [10]:
graph = HanaRdfGraph(
 connection=connection,
 graph_uri="http://kg.demo.sap.com/movies",
 ontology_uri="http://kg.demo.sap.com/movies_ontology"
)

3. **Local RDF file** (`ontology_local_file` + `ontology_local_file_format`): Load the schema from a local RDF ontology file. Supported RDF formats are `Turtle`, `RDF/XML`, `JSON-LD`, `N-Triples`, `Notation-3`, `Trig`, `Trix`, `N-Quads`.


In [11]:
graph = HanaRdfGraph(
 connection=connection,
 graph_uri="http://kg.demo.sap.com/movies",
 ontology_local_file="movies_ontology.ttl",
 ontology_local_file_format="turtle"
)

4. **Auto-extract ontology** (`auto_extract_ontology=True`): Infer schema information directly from your instance data.

In [12]:
graph_with_auto_extracted_ontology = HanaRdfGraph(
 connection=connection,
 graph_uri="http://kg.demo.sap.com/movies",
 auto_extract_ontology=True
)

> **Note**: Auto-extraction is **not** recommended for production—it omits important triples like `rdfs:label`, `rdfs:comment`, and `rdfs:subClassOf` in general.

## Executing SPARQL Queries

You can use the `query()` method to execute arbitrary SPARQL queries (`SELECT`, `ASK`, `CONSTRUCT`, etc.) on the data graph.  


The following query retrieves the top 10 movies with the highest number of contributors:

In [13]:
query = """
PREFIX kg: <http://kg.demo.sap.com/>
SELECT ?movieTitle (COUNT(?crewmember) AS ?contributorCount)

FROM <http://kg.demo.sap.com/movies>
WHERE {
    ?crewmember kg:contributed_to ?movie .
    ?movie kg:title ?movieTitle .
}
GROUP BY ?movieTitle
ORDER BY DESC(?contributorCount)
LIMIT 10
"""
top10 = graph.query(query)
print(top10)

movieTitle,contributorCount
Jurassic World,430
15 Minutes,329
The Wolf of Wall Street,239
"Monsters, Inc.",232
The Day After Tomorrow,223
The Core,219
The Dark Knight Rises,210
The X Files: I Want to Believe,195
V for Vendetta,190
The Chronicles of Riddick,186



## Question Answering with `HanaSparqlQAChain`

`HanaSparqlQAChain` ties together:

1. **Schema-aware SPARQL generation**  
2. **Query execution** against SAP HANA  
3. **Natural-language answer formatting**


### Initialization

You need:

- An **LLM** to generate and interpret queries  
- A **`HanaRdfGraph`** (with connection, `graph_uri`, and ontology)

In [14]:
from langchain_openai import ChatOpenAI  # or your chosen LLM
llm = ChatOpenAI(model="gpt-4o")

qa_chain = HanaSparqlQAChain.from_llm(
    llm=llm,
    graph=graph,
    allow_dangerous_requests=True,
    verbose=True
)

### Pipeline Overview

1. **SPARQL Generation**  
   - Uses `SPARQL_GENERATION_SELECT_PROMPT`  
   - Inputs:  
     - `schema` (Turtle from `graph.get_schema`)  
     - `prompt` (user’s question)  
2. **Query Post-processing**  
   - Extracts the SPARQL code from the llm output.
   - Inject `FROM <graph_uri>` if missing  
   - Ensure required common prefixes are declared (`rdf:`, `rdfs:`, `owl:`, `xsd:`)  
3. **Execution**  
   - Calls `graph.query(generated_sparql)`  
4. **Answer Formulation**  
   - Uses `SPARQL_QA_PROMPT`  
   - Inputs:  
     - `context` (raw query results)  
     - `prompt` (original question)  

### Prompt Templates


#### "SPARQL Generation" prompt

The `sparql_generation_prompt` is used to guide the LLM in generating a SPARQL query from the user question and the provided schema.

Default template:
  ````python
  SPARQL_GENERATION_SELECT_TEMPLATE = """Task: Generate a SPARQL SELECT statement for querying a graph database.
    For instance, to find all email addresses of John Doe, the following query in backticks would be suitable:
    ```
    PREFIX foaf: <http://xmlns.com/foaf/0.1/>
    SELECT ?email
    WHERE {{
        ?person foaf:name "John Doe" .
        ?person foaf:mbox ?email .
    }}
    ```
    Instructions:
    Use only the node types and properties provided in the schema.
    Do not use any node types and properties that are not explicitly provided.
    Include all necessary prefixes.
    Schema:
    {schema}
    Do not respond to any questions that ask for anything else than for you to construct a SPARQL query.
    Do not include any text except the SPARQL query generated.
    Please pay attention to providing the subject, predicate, and object in the correct order.
    Ensure that every variable referenced in any clause (such as SELECT, ORDER BY, GROUP BY, etc.) is explicitly defined in the WHERE clause, either by being used as a subject, predicate, or object in a triple pattern, or through a BIND statement. 
    Do not include any variables in those clauses unless they are defined in the WHERE clause.
    
    The question is:
    {prompt}"""
    SPARQL_GENERATION_SELECT_PROMPT = PromptTemplate(
        input_variables=["schema", "prompt"], template=SPARQL_GENERATION_SELECT_TEMPLATE
    )
  ````


#### Answering prompt

The `qa_prompt` instructs the LLM to create a natural language answer based solely on the database results.
  
Default template:
  ````python
  SPARQL_QA_TEMPLATE = """Task: Generate a natural language response from the results of a SPARQL query.
    You are an assistant that creates well-written and human understandable answers.
    The information part contains the information provided, which you can use to construct an answer.
    The information provided is authoritative, you must never doubt it or try to use your internal knowledge to correct it.
    Make your response sound like the information is coming from an AI assistant, but don't add any information. 
    Don't use internal knowledge to answer the question, just say you don't know if no information is available.
    Information:
    {context}
    
    Question: {prompt}
    Helpful Answer:"""
    SPARQL_QA_PROMPT = PromptTemplate(
        input_variables=["context", "prompt"], template=SPARQL_QA_TEMPLATE
    )
  ````

### Customizing Prompts

You can override the defaults at initialization:

```python
qa_chain = HanaSparqlQAChain.from_llm(
    llm=llm,
    graph=graph,
    allow_dangerous_requests=True,
    verbose=True,
    sparql_generation_prompt=YOUR_SPARQL_PROMPT,
    qa_prompt=YOUR_QA_PROMPT
)
```

Or swap them afterward:

```python
qa_chain.sparql_generation_chain.prompt = YOUR_SPARQL_PROMPT
qa_chain.qa_chain.prompt              = YOUR_QA_PROMPT
```

> - `sparql_generation_prompt` must have the input variables: `["schema", "prompt"]`
> - `qa_prompt` must have the input variables: `["context", "prompt"]`